# Hyperliquid consolidated account analytics

Combines balances, account history, fills, funding, positions, and PnL across the configured Hyperliquid wallets. Combined account-value history is shown only over the dates available for every wallet; charts show the oldest wallet's account history before shared coverage begins, then the combined account value.

Daily rollups and calendar-period cutoffs use `America/New_York`. Previously saved UTC-aggregated daily totals are mapped to Eastern date labels when loaded; their original intraday split cannot be reconstructed.


## Wallets and implementation

Edit `WALLET_ADDRESSES` here. Both addresses found in the single-wallet notebook are included: the active address and the previously commented address.


In [185]:
import json
from pathlib import Path

WALLET_ADDRESSES = [
    '0xf3fb3598bFfA608d1e7CF2837d74F02B1191c248',
    '0x7532dA34edA8f3e383bC480d50e7A7d3b12c07f2',
]

RISK_BUDGET_PCT = 5.0
ATR_PERIOD = 14
ATR_STOP_MULTIPLE = 1.25
MARGIN_RESERVE_PCT = 20.0
namespace={'__name__':'hyperliquid_multi_account_analytics'}
implementation_paths=[
    Path('hyperliquid_account_analytics_implementation.ipynb'),
    Path('hyperliquid_multi_account_analytics_implementation.ipynb'),
]
for implementation_path in implementation_paths:
    implementation=json.loads(implementation_path.read_text(encoding='utf-8'))
    for cell in implementation['cells']:
        if cell['cell_type']=='code':
            source=''.join(cell['source'])
            if source.strip():
                exec(compile(source,f'{implementation_path} cell','exec'),namespace)
MultiWalletAnalytics=namespace['MultiWalletAnalytics']
analytics=MultiWalletAnalytics(WALLET_ADDRESSES,data_dir='data')
analytics.position_risk_budget_pct=RISK_BUDGET_PCT
analytics.position_atr_period=ATR_PERIOD
analytics.position_atr_stop_multiple=ATR_STOP_MULTIPLE
analytics.margin_reserve_pct=MARGIN_RESERVE_PCT


## Fetch and consolidate wallet data

Fetch each wallet once and prepare combined account history and fills.


In [186]:
%%capture
_ = analytics.fetch_data()


## Account / PnL

Combined account and risk summary. Daily PnL and calendar-period cutoffs use US Eastern time.


In [187]:
analytics.show_account_pnl()


=== ACCOUNT / PNL AND RISK ===


Metric,Value
Account value,"$2,918.78"
Portfolio value,"$3,470.60 (combined live wallet value)"
Account 1D (ET),2.39%
Realized today (ET),$16.14
Realized today (% of account value),0.55%
Unrealized PnL,$-180.14
Unrealized PnL (% of account value),-6.17%
Unrealized 1D change (ET),n/a
Average realized PnL / day,$-0.96
Average unrealized PnL change / day,n/a


## Account charts

Account value and PnL charts. Before all wallets have saved history, account value and weekly returns use the oldest wallet alone; from shared coverage onward they use the combined accounts.


In [188]:
analytics.show_account_charts()


Weekly return chart omits 2 return(s) below -100%;                     other data and stats are unchanged.


## Account vs BTC

Daily account-vs-BTC comparison uses the oldest wallet before shared history begins, then the combined accounts. The lifetime realized-PnL comparison pools the available wallet history.


In [189]:
analytics.show_account_vs_btc()


Lifetime account series is anchored to current account value; it includes fee-net realized trades and perp funding, but excludes historical cash flows and unrealized PnL.


## Traded volume vs realized PnL chart

Daily fill volume and realized PnL pooled across wallets.


In [190]:
analytics.show_volume_chart()


## Open positions

Show each wallet position separately, followed by fixed-fractional sizing grouped by asset and side and an aggregate summary. The summary includes cumulative open-position funding as a percentage of combined account equity and total position value. Reserve-adjusted available margin is combined account equity less the configured reserve; the summary shows that capacity, margin used, remaining available margin in dollars and as a percentage of that capacity, and position value as a percentage of equity. Set `MARGIN_RESERVE_PCT` (20% by default) in the setup cell to change the reserve. The position table includes funding and fees since open as dollar amounts, plus fees since open as percentages of wallet equity and position value and position value as a percentage of wallet account value. Fee ratios are available for complete USDC-denominated fee history. Set `RISK_BUDGET_PCT` (5% per asset), `ATR_PERIOD` (14 completed daily candles), and `ATR_STOP_MULTIPLE` (1.25) in the setup cell. Stops are placed 1.25 ATR from the current mark, and take-profit is set the same distance in the favorable direction (1:1 risk/reward). Same-side positions across wallets are combined; long and short rows remain separate. If both sides are open for one asset, they share that asset's risk budget equally.


In [191]:
analytics.show_open_positions()


Wallet,Coin,Side,Liq %,ROE,uPnL,Funding (since open),Funding (1h),Funding APR,Entry,Mark,Pos,Margin,Margin Type,Margin %,Fees,Fees / Equity (%),Fees / Position (%),Position / Account (%),Liq Px,Lev.,Sz
c248,IMX,SHORT,63.62%,1.77%,$3.51,$0.46,0.001250%,10.95%,0.12332,0.1229,$988,$198,Cross,48.61%,-$0.14,0.016%,0.014%,110.01%,0.20109,5.00x,"-8,040.50"
07f2,TRX,SHORT,197.61%,0.16%,$0.11,$-0.00,-0.000183%,-1.60%,0.313,0.31295,$697,$70,Cross,7.61%,-$0.10,0.004%,0.014%,27.08%,0.93138,10.00x,"-2,226.00"
07f2,PENGU,SHORT,208.16%,1.54%,$1.95,$0.19,0.001250%,10.95%,0.007292,0.00727,$631,$126,Cross,13.80%,-$0.09,0.004%,0.014%,24.54%,0.022403,5.00x,"-86,832.00"
07f2,xyz:GOOGL,LONG,n/a,-52.82%,-$102.48,-$2.24,0.002641%,23.13%,343.02,282.64,$480,$160,Cross,17.48%,-$0.05,0.002%,0.010%,18.65%,0,3.00x,1.70
07f2,MELANIA,SHORT,283.51%,-0.59%,-$0.85,$-0.00,0.001250%,10.95%,0.07686,0.07701,$437,$146,Cross,15.92%,-$0.06,0.002%,0.014%,16.99%,0.29534,3.00x,"-5,674.80"
07f2,io:PRL,SHORT,11.22%,-10.64%,-$14.53,$4.79,0.023706%,207.66%,1.50057,1.5538,$424,$126,Isolated,13.80%,-$0.04,0.001%,0.009%,16.49%,1.7282,3.36x,-273.00
c248,AVAX,SHORT,253.50%,3.35%,$2.93,$0.03,0.001250%,10.95%,8.7597,8.662,$260,$87,Cross,21.31%,-$0.04,0.004%,0.015%,28.93%,30.62,3.00x,-30.00
07f2,ADA,SHORT,658.13%,1.94%,$1.02,-$0.08,0.001250%,10.95%,0.193547,0.19261,$209,$52,Cross,5.72%,-$0.47,0.018%,0.225%,8.13%,1.4602,4.00x,"-1,086.00"
c248,INIT,SHORT,12.29%,-5.35%,-$3.62,-$0.00,-0.001016%,-8.90%,0.18359,0.18682,$206,$64,Isolated,15.74%,-$0.05,0.005%,0.024%,22.99%,0.20979,3.23x,"-1,105.00"
07f2,xyz:RDDT,SHORT,3.80%,-11.33%,-$2.18,$0.36,-0.005048%,-44.22%,92.041,93.084,$195,$17,Isolated,1.91%,-$0.02,0.001%,0.009%,7.56%,96.622,11.12x,-2.09


Funding and fees are cumulative since the position was opened. Fee ratios are shown only for complete USDC-denominated fees; otherwise they are n/a.
Fixed-fractional sizing: risk budget 5.00% of combined account value per asset; stop distance 1.25 x daily ATR(14). Opposing sides, when both open, split the asset budget equally.
ATR stop-distance colors: green <2%, orange 2-5%, red >5%; red mark means the position is out of the money; stop prices are highlighted red.


Asset,Side,ATR (%),ATR Period Used,ATR Stop Distance (%),Avg Entry,Mark,Stop,Take Profit,Open - Target (coins),Open - Target (%),Open Risk ($)
ADA,SHORT,6.60%,14 days,8.25%,0.193547,0.19261,0.2085,0.17672,"9,835.41",-90.06%,$17.26
AERO,LONG,10.30%,14 days,12.88%,0.832395,0.62853,0.54759,0.70947,"-2,024.93",-94.45%,$9.63
AVAX,SHORT,6.34%,14 days,7.93%,8.7597,8.662,9.3488,7.9752,222.68,-88.13%,$20.60
IMX,SHORT,9.13%,14 days,11.42%,0.12332,0.1229,0.13693,0.10887,"4,326.33",-34.98%,$112.82
INIT,SHORT,14.14%,14 days,17.67%,0.18359,0.18682,0.21983,0.15381,"4,151.34",-78.98%,$36.48
LIT,SHORT,11.76%,14 days,14.70%,3.2512,3.153854,3.6174,2.6903,365.371,-97.60%,$4.17
MELANIA,SHORT,7.30%,14 days,9.12%,0.07686,0.07701,0.084035,0.069985,"19,026.7",-77.03%,$39.87
PENGU,SHORT,8.45%,14 days,10.56%,0.007292,0.00727,0.0080378,0.0065022,"139,190",-61.58%,$66.67
PROVE,SHORT,6.79%,14 days,8.49%,0.203334,0.19978,0.21675,0.18281,"9,684.28",-94.69%,$9.21
RSR,SHORT,11.60%,14 days,14.50%,0.001385,0.001471,0.0016844,0.0012576,"718,536",-88.35%,$20.22


Metric,Value
Wallets,2
Position value,"$5,557"
uPnL,$-180.14
ROE,-13.62%
Margin,"$1,323"
Margin % bal,36.23%
Leverage (Pos / Margin),4.20x
"Funding (open positions, since open)",$3.88
Funding / Equity (%),0.112%
Funding / Position (%),0.070%


ROE is unrealized PnL divided by used margin; a negative ROE means the open positions are currently at an unrealized loss relative to the margin supporting them.


## Realized PnL statistics

Pool spot/perp fills and signed funding across wallets.


In [192]:
analytics.show_realized_pnl()


=== REALIZED SPOT ===


Metric,Value
Realized PnL,$22.35 (0.64%)
24H,$0.00 (0.00%)
7D,$0.00 (0.00%)
30D,$22.35 (0.64%)
QTD,$22.35 (0.64%)
YTD,$22.35 (0.64%)
W,3
Win %,100.00%
Avg W,$7.45
L,0


=== REALIZED PERP ===


Metric,Value
Realized PnL,$-60.09 (-1.73%)
24H,$5.57 (0.16%)
7D,$357.40 (10.30%)
30D,$463.92 (13.37%)
QTD,$463.84 (13.36%)
YTD,$463.71 (13.36%)
W,187
Win %,68.50%
Avg W,$8.49
L,86



=== PERP PNL BREAKDOWN ===


Component,Amount ($),Contribution (% of |net PnL|)
Trading,$99.35,165.34%
Fees,$-106.21,-176.77%
Funding,$-53.22,-88.57%
Net perp PnL,$-60.09,


Liquidations: 0 of 273 closes (0.00%) | liquidation PnL: $0.00.


## Traded volume vs PnL

Combined turnover efficiency and correlations.


In [193]:
analytics.show_volume_summary()


=== TRADED VOLUME VS REALIZED PNL (ALL WALLETS) ===


Market,Volume ($),Realized PnL ($),Realized PnL / Volume (%),PnL / $1k volume ($),Active days,Daily Pearson r,Daily Spearman r
Perps,"$251,788.27",$-6.87,-0.00%,$-0.03,57,0.102,0.293
Spot,"$1,397.53",$22.35,1.60%,$16.00,5,-0.509,-0.462


## PnL attribution

Combined realized and open-position PnL by asset.


In [194]:
analytics.show_pnl_attribution()


=== CONSOLIDATED PNL ATTRIBUTION ===


Type,Asset,Closes,Realized,Realized (% gross),uPnL,uPnL (% gross),Total PnL,Total (% gross)
Perp,BTC,11,$-474.42,-18.85%,$0.00,0.00%,$-474.42,-17.76%
Perp,kBONK,3,$-397.06,-15.78%,$0.00,0.00%,$-397.06,-14.86%
Perp,ZORA,21,$304.49,12.10%,$0.00,0.00%,$304.49,11.40%
Perp,KAITO,4,$-193.84,-7.70%,$0.00,0.00%,$-193.84,-7.26%
Perp,HYPE,11,$146.54,5.82%,$0.00,0.00%,$146.54,5.49%
Perp,ETH,14,$116.62,4.63%,$0.00,0.00%,$116.62,4.37%
Perp,VVV,40,$116.26,4.62%,$0.00,0.00%,$116.26,4.35%
Perp,xyz:GOOGL,0,$-2.57,-0.10%,$-102.48,-49.25%,$-105.05,-3.93%
Perp,io:ANTH,2,$-86.69,-3.44%,$0.00,0.00%,$-86.69,-3.25%
Perp,AAVE,4,$54.17,2.15%,$0.00,0.00%,$54.17,2.03%


Net attributed PnL across wallets: $-217.98.
